In [54]:
import pandas as pd
import numpy as np
from scipy.special import expit
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
# ============================
# 0. Setup & Cleanup
# ============================
np.random.seed(42)

df_original = pd.read_csv("Student Perfomance Prediction\\data\\raw\\test.csv")
print("Original Shape:", df_original.shape)

if 'Student_ID' not in df_original.columns:
    df_original['Student_ID'] = range(1, len(df_original) + 1)

df_original = df_original.dropna(subset=[
    'TestScore_Math', 'TestScore_Science', 'TestScore_Reading',
    'GPA', 'StudyHours', 'InternetAccess', 'Locale', 'AttendanceRate'
])

for col in ['TestScore_Math', 'TestScore_Science', 'TestScore_Reading', 'GPA', 'StudyHours', 'InternetAccess', 'AttendanceRate']:
    df_original[col] = pd.to_numeric(df_original[col])

df_original['Locale'] = df_original['Locale'].astype(str).str.strip()

n_students = len(df_original)

# ============================
# 1. Student-Level Traits 
# ============================

# 1.1 Latent Academic Ability
df_original['Latent_Ability'] = np.random.normal(0, 1, n_students)

# 1.2 Archetypes
archetypes = ['elite', 'fading_star', 'late_bloomer', 'disengaged', 'pragmatist', 'vulnerable']
probs = [0.12, 0.13, 0.15, 0.20, 0.18, 0.22]

df_original['Archetype'] = np.random.choice(archetypes, size=n_students, p=probs)

conditions = [
    df_original['Archetype'] == 'elite',
    df_original['Archetype'] == 'fading_star',
    df_original['Archetype'] == 'late_bloomer',
    df_original['Archetype'] == 'disengaged',
    df_original['Archetype'] == 'pragmatist',
    df_original['Archetype'] == 'vulnerable'
]

# ============================
# 1.3 Base Behavioral Features
# ============================

df_original['Base_StudyHours'] = np.select(conditions, [4.5, 3.5, 3.0, 1.8, 2.7, 2.2])
df_original['Base_Attendance'] = np.select(conditions, [0.92, 0.85, 0.80, 0.65, 0.78, 0.70])
df_original['Base_Assignment'] = np.select(conditions, [0.95, 0.85, 0.80, 0.60, 0.75, 0.68])
df_original['Base_Engagement'] = np.select(conditions, [0.85, 0.70, 0.75, 0.50, 0.68, 0.60])
df_original['Base_Social'] = np.select(conditions, [0.20, 0.40, 0.30, 0.70, 0.40, 0.45])
df_original['Base_Effort'] = np.select(conditions, [0.80, 0.65, 0.85, 0.45, 0.65, 0.55])

# ============================
# 1.4 Psychological + Environment Traits
# ============================

df_original['study_environment_score'] = np.select(
    conditions,
    [8.5, 7.5, 7.0, 5.5, 7.2, 6.2]
) + np.random.normal(0, 0.8, n_students)

df_original['first_generation_college_flag'] = np.select(
    conditions,
    [0, 0, 0, 1, 0, 1]
)

df_original['procrastination_tendency'] = np.select(
    conditions,
    [2.0, 3.5, 3.0, 7.0, 4.0, 5.5]
) + np.random.normal(0, 0.7, n_students)

df_original['stress_survey_score'] = np.select(
    conditions,
    [4.5, 6.0, 5.5, 7.0, 5.8, 7.5]
) + np.random.normal(0, 0.8, n_students)

df_original['exam_anxiety_score'] = np.select(
    conditions,
    [3.0, 5.0, 4.5, 6.5, 4.5, 7.0]
) + np.random.normal(0, 0.7, n_students)

df_original['motivation_survey_score'] = np.select(
    conditions,
    [8.5, 6.0, 8.0, 3.5, 6.5, 5.5]
) + np.random.normal(0, 0.7, n_students)

df_original['sleep_hours'] = np.select(
    conditions,
    [7.5, 6.8, 7.2, 6.0, 6.8, 6.5]
) + np.random.normal(0, 0.5, n_students)

df_original['self_efficacy_score'] = np.select(
    conditions,
    [8.5, 6.0, 7.5, 4.0, 6.5, 5.0]
) + np.random.normal(0, 0.7, n_students)

df_original['goal_clarity_survey_score'] = np.select(
    conditions,
    [8.5, 5.5, 7.8, 3.5, 6.5, 5.0]
) + np.random.normal(0, 0.7, n_students)

# ============================
# Static Environment
# ============================

locale_map = {'Rural': 3, 'Town': 2, 'Suburban': 2, 'City': 1}
df_original['Commute_Strain'] = df_original['Locale'].map(locale_map).fillna(2)

# ============================
# 2. Longitudinal Expansion
# ============================

df_expanded = df_original.loc[df_original.index.repeat(8)].reset_index(drop=True)
df_expanded['Semester_ID'] = np.tile(range(1, 9), n_students)

# ============================
# 3. Recursive Semester Loop 
# ============================

df_expanded['Prev_GPA'] = 0.0
df_expanded['Final_Sem_GPA'] = 0.0
df_expanded['Backlog_Count'] = 0

for sem in range(1, 9):

    mask = df_expanded['Semester_ID'] == sem
    n_sem_students = mask.sum()

    latent = df_expanded.loc[mask, 'Latent_Ability'].values
    archs = df_expanded.loc[mask, 'Archetype'].values

    # 1. Reduced Behavioral Noise (smoother habits)
    sem_study = np.clip(df_expanded.loc[mask, 'Base_StudyHours'].values + np.random.normal(0, 0.2, n_sem_students), 0, 10)
    sem_attnd = np.clip(df_expanded.loc[mask, 'Base_Attendance'].values + np.random.normal(0, 0.03, n_sem_students), 0.1, 1)
    sem_assgn = np.clip(df_expanded.loc[mask, 'Base_Assignment'].values + np.random.normal(0, 0.03, n_sem_students), 0, 1)
    sem_soc = np.clip(df_expanded.loc[mask, 'Base_Social'].values + np.random.normal(0, 0.05, n_sem_students), 0, 1)
    sem_effrt = np.clip(df_expanded.loc[mask, 'Base_Effort'].values + np.random.normal(0, 0.05, n_sem_students), 0, 1)

    df_expanded.loc[mask, 'StudyHours'] = sem_study
    df_expanded.loc[mask, 'AttendanceRate'] = sem_attnd
    df_expanded.loc[mask, 'Assignment_Ratio'] = sem_assgn
    df_expanded.loc[mask, 'Social_Distraction'] = sem_soc
    df_expanded.loc[mask, 'Effort_Score'] = sem_effrt

    # 2. Archetype Drift (Smooth gradual changes over semesters)
    trend = np.zeros(n_sem_students)
    trend[archs == 'fading_star'] = -0.15 * sem      # Gradual decline
    trend[archs == 'late_bloomer'] = 0.20 * sem      # Gradual rise
    trend[archs == 'disengaged'] = -0.08 * sem       # Slow decay from lack of effort
    trend[archs == 'elite'] = 0.05 * sem             # Slight compounding mastery

    # 3. Smoothed IA Scores (Reduced noise, anchored to latent ability & trend)
    ia1_base = 8.0 + (latent * 1.5) + (sem_study * 0.8) + (sem_attnd * 5.0) + trend
    ia1 = ia1_base + np.random.normal(0, 0.8, n_sem_students) # Noise reduced from 2.0 to 0.8
    df_expanded.loc[mask, 'IA1_Score'] = np.clip(ia1, 0, 20).round(1)

    ia_improvement = (sem_effrt * 2.5 - sem_soc * 1.5 + np.random.normal(0, 0.5, n_sem_students))
    df_expanded.loc[mask, 'IA_Improvement'] = ia_improvement.round(2)

    ia2 = ia1 + ia_improvement
    df_expanded.loc[mask, 'IA2_Score'] = np.clip(ia2, 0, 20).round(1)

    df_expanded.loc[mask, 'Performance_Delta'] = df_expanded.loc[mask, 'IA2_Score'] - df_expanded.loc[mask, 'IA1_Score']
    ia_avg = (df_expanded.loc[mask, 'IA1_Score'] + df_expanded.loc[mask, 'IA2_Score']) / 2
    df_expanded.loc[mask, 'IA_Average'] = ia_avg

    # 4. Target GPA (What the student "deserves" this semester based on raw behavior)
    target_gpa = 2.0 + (latent * 0.8) + (ia_avg / 4) + (sem_study * 0.3) + (sem_attnd * 3.0) - (sem_soc * 1.0) + trend
    target_gpa = np.clip(target_gpa, 0, 10)
    
    # ... inside the sem loop, after target_gpa calculation ...

    # ==========================================
    # 1. Apply Burnout (Starts after Sem 5)
    # ==========================================
    # We apply this to the target_gpa so it represents 
    # the student's actual performance capacity dropping.
    burnout_penalty = np.maximum(0, sem - 5) * 0.15 
    target_gpa = np.clip(target_gpa - burnout_penalty, 0, 10)

    # 5. Calculate Final GPA with Historical Inertia
    if sem == 1:
        # First semester is just their target + slight noise
        final_gpa = target_gpa + np.random.normal(0, 0.4, n_sem_students)
        df_expanded.loc[mask, 'Prev_GPA'] = np.clip(final_gpa, 0, 10).round(2)
        df_expanded.loc[mask, 'Backlog_Count'] = 0
        prev_backlogs = np.zeros(n_sem_students)
    else:
        prev_mask = df_expanded['Semester_ID'] == (sem - 1)
        prev_gpa = df_expanded.loc[prev_mask, 'Final_Sem_GPA'].values
        prev_backlogs = df_expanded.loc[prev_mask, 'Backlog_Count'].values

        df_expanded.loc[mask, 'Prev_GPA'] = prev_gpa

        # Backlog logic
        fail_prob = np.clip((6 - prev_gpa) / 6, 0.05, 0.6)
        new_fails = np.random.binomial(2, fail_prob)
        cleared = np.random.binomial(prev_backlogs.astype(int), 0.5)
        df_expanded.loc[mask, 'Backlog_Count'] = np.clip(prev_backlogs + new_fails - cleared, 0, 15)
        
        # ==========================================
        # Archetype-dependent inertia
        # ==========================================

        inertia = np.full(n_sem_students, 0.70)

        inertia[archs == 'elite'] = 0.80
        inertia[archs == 'fading_star'] = 0.65
        inertia[archs == 'late_bloomer'] = 0.65
        inertia[archs == 'disengaged'] = 0.55
        inertia[archs == 'pragmatist'] = 0.70
        inertia[archs == 'vulnerable'] = 0.60

        # EMA Formula: 70% inertia from previous GPA, 30% new target behavior
        semester_noise = np.random.normal(0, 0.25, n_sem_students) # Noise heavily reduced
        final_gpa = (inertia * prev_gpa + (1 - inertia) * target_gpa + semester_noise)
    df_expanded.loc[mask, 'Final_Sem_GPA'] = np.clip(final_gpa, 0, 10).round(2)

# ============================
# 4. Dropout Probability
# ============================

dropout_prob = (
    0.4*(1-df_expanded['Final_Sem_GPA']/10) +
    0.3*(1-df_expanded['AttendanceRate']) +
    0.2*df_expanded['Social_Distraction'] +
    0.05*df_expanded['Backlog_Count'] +
    0.05*df_expanded['Commute_Strain']
)

df_expanded['Dropout_Probability'] = (expit(dropout_prob*5-2)*0.95).round(2)
df_expanded['Dropout_Flag'] = (df_expanded['Dropout_Probability'] > 0.65).astype(int)

# ============================
# 5. Cleanup
# ============================

cols_to_drop = [
'Base_StudyHours',
'Base_Attendance',
'Base_Assignment',
'Base_Engagement',
'Base_Social',
'Base_Effort'
]

df_final = df_expanded.drop(columns=cols_to_drop)

print("Final Expanded Shape:", df_final.shape)

columns_to_view = [
'Student_ID',
'Semester_ID',
'Archetype',
'Prev_GPA',
'IA1_Score',
'IA2_Score',
'Final_Sem_GPA',
'Backlog_Count',
'Dropout_Probability',
'Dropout_Flag'
]

print("\n--- Sample Timeline for Student ID 1 ---")
print(df_final[df_final['Student_ID']==1][columns_to_view])

Original Shape: (999997, 21)
Final Expanded Shape: (7999976, 48)

--- Sample Timeline for Student ID 1 ---
   Student_ID  Semester_ID   Archetype  Prev_GPA  IA1_Score  IA2_Score  \
0           1            1  disengaged      7.41       15.9       16.2   
1           1            2  disengaged      7.41       11.8       11.6   
2           1            3  disengaged      7.64       13.0       13.1   
3           1            4  disengaged      7.89       14.8       15.5   
4           1            5  disengaged      7.33       12.7       12.9   
5           1            6  disengaged      7.56       12.3       12.6   
6           1            7  disengaged      7.01       13.6       13.4   
7           1            8  disengaged      6.80       13.1       12.0   

   Final_Sem_GPA  Backlog_Count  Dropout_Probability  Dropout_Flag  
0           7.41              0                 0.49             0  
1           7.64              0                 0.47             0  
2           7.89   

In [4]:
# Columns to stratify on
strat_cols = ['Gender', 'Race', 'SES_Quartile', 'SchoolType']

# Create a combined stratification column
df_final['strata'] = df_final[strat_cols].astype(str).agg('_'.join, axis=1)

# Perform stratified sampling: 300k rows
df_sample, _ = train_test_split(
    df_final,
    train_size=300_000,
    stratify=df_final['strata'],   # preserves proportions of all category combinations
    random_state=42
)

# Drop the temporary 'strata' column
df_sample = df_sample.drop(columns=['strata'])

# Quick checks
print(len(df_sample))  
print(df_sample['Gender'].value_counts())
print(df_sample['Race'].value_counts())
print(df_sample['SES_Quartile'].value_counts())
print(df_sample['SchoolType'].value_counts()) 

300000
Gender
Female    153029
Male      146971
Name: count, dtype: int64
Race
White          131914
Hispanic        87100
Black           44904
Two-or-more     15025
Asian           14980
Other            6077
Name: count, dtype: int64
SES_Quartile
3    75057
4    74994
2    74991
1    74958
Name: count, dtype: int64
SchoolType
Public     253327
Private     46673
Name: count, dtype: int64


In [5]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 300000 entries, 2501236 to 5406787
Data columns (total 48 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Age                            300000 non-null  int64  
 1   Grade                          300000 non-null  int64  
 2   Gender                         300000 non-null  object 
 3   Race                           300000 non-null  object 
 4   SES_Quartile                   300000 non-null  int64  
 5   ParentalEducation              300000 non-null  object 
 6   SchoolType                     300000 non-null  object 
 7   Locale                         300000 non-null  object 
 8   TestScore_Math                 300000 non-null  float64
 9   TestScore_Reading              300000 non-null  float64
 10  TestScore_Science              300000 non-null  float64
 11  GPA                            300000 non-null  float64
 12  AttendanceRate              

In [72]:
df_sample.columns

Index(['Age', 'Grade', 'Gender', 'SES_Quartile', 'ParentalEducation',
       'SchoolType', 'GPA', 'AttendanceRate', 'StudyHours', 'InternetAccess',
       'Extracurricular', 'PartTimeJob', 'ParentSupport', 'Romantic',
       'FreeTime', 'GoOut', 'Student_ID', 'Latent_Ability',
       'study_environment_score', 'first_generation_college_flag',
       'procrastination_tendency', 'stress_survey_score', 'exam_anxiety_score',
       'motivation_survey_score', 'sleep_hours', 'self_efficacy_score',
       'goal_clarity_survey_score', 'Commute_Strain', 'Semester_ID',
       'Prev_GPA', 'Final_Sem_GPA', 'Backlog_Count', 'Assignment_Ratio',
       'Social_Distraction', 'Effort_Score', 'IA1_Score', 'IA_Improvement',
       'IA2_Score', 'Performance_Delta', 'IA_Average', 'Dropout_Probability',
       'Dropout_Flag', 'Locale_Rural', 'Locale_Suburban', 'Locale_Town',
       'Race_Black', 'Race_Hispanic', 'Race_Other', 'Race_Two-or-more',
       'Race_White', 'Archetype_elite', 'Archetype_fading_star

In [6]:
for col in df_sample.select_dtypes(include=['object']).columns:
    print(df_sample[col].unique())

['Female' 'Male']
['White' 'Hispanic' 'Asian' 'Two-or-more' 'Black' 'Other']
['HS' 'Bachelors+' '<HS' 'SomeCollege']
['Public' 'Private']
['Town' 'Rural' 'Suburban' 'City']
['vulnerable' 'pragmatist' 'fading_star' 'late_bloomer' 'elite'
 'disengaged']


In [7]:
def mapping_features(X):
  gender_map = {'Male': 0, 'Female': 1}
  X['Gender'] = X['Gender'].map(gender_map)
  education_map = {'<HS': 0, 'HS': 1, 'SomeCollege': 2, 'Bachelors+': 3}
  X['ParentalEducation'] = X['ParentalEducation'].map(education_map)
  X['SchoolType'] = X['SchoolType'].map({'Public': 0, 'Private': 1})
  X = pd.get_dummies(X, columns=['Locale'], drop_first=True)
  X = pd.get_dummies(X, columns=['Race'], drop_first=True) 
  X = pd.get_dummies(X, columns=['Archetype'], drop_first=True) 
  return X
df_sample = mapping_features(df_sample)

In [8]:
df_sample.columns

Index(['Age', 'Grade', 'Gender', 'SES_Quartile', 'ParentalEducation',
       'SchoolType', 'TestScore_Math', 'TestScore_Reading',
       'TestScore_Science', 'GPA', 'AttendanceRate', 'StudyHours',
       'InternetAccess', 'Extracurricular', 'PartTimeJob', 'ParentSupport',
       'Romantic', 'FreeTime', 'GoOut', 'Student_ID', 'Latent_Ability',
       'study_environment_score', 'first_generation_college_flag',
       'procrastination_tendency', 'stress_survey_score', 'exam_anxiety_score',
       'motivation_survey_score', 'sleep_hours', 'self_efficacy_score',
       'goal_clarity_survey_score', 'Commute_Strain', 'Semester_ID',
       'Prev_GPA', 'Final_Sem_GPA', 'Backlog_Count', 'Assignment_Ratio',
       'Social_Distraction', 'Effort_Score', 'IA1_Score', 'IA_Improvement',
       'IA2_Score', 'Performance_Delta', 'IA_Average', 'Dropout_Probability',
       'Dropout_Flag', 'Locale_Rural', 'Locale_Suburban', 'Locale_Town',
       'Race_Black', 'Race_Hispanic', 'Race_Other', 'Race_Two-or-mo

In [9]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 300000 entries, 2501236 to 5406787
Data columns (total 58 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Age                            300000 non-null  int64  
 1   Grade                          300000 non-null  int64  
 2   Gender                         300000 non-null  int64  
 3   SES_Quartile                   300000 non-null  int64  
 4   ParentalEducation              300000 non-null  int64  
 5   SchoolType                     300000 non-null  int64  
 6   TestScore_Math                 300000 non-null  float64
 7   TestScore_Reading              300000 non-null  float64
 8   TestScore_Science              300000 non-null  float64
 9   GPA                            300000 non-null  float64
 10  AttendanceRate                 300000 non-null  float64
 11  StudyHours                     300000 non-null  float64
 12  InternetAccess              

In [12]:
def feature_engineering(df):
    # --- Original Features ---
    df['Academic_Performance'] = (df['TestScore_Math'] + df['TestScore_Science'] + df['TestScore_Reading']) / 3
    df['Engagement_Index'] = (df['AttendanceRate'] * 0.5 + df['StudyHours'] * 0.3 + df['Extracurricular'] * 0.2)
    df['Support_Score'] = (df['SES_Quartile'] + df['ParentalEducation'] + df['ParentSupport']) 
    df["IA_Average"] = (df["IA1_Score"] + df["IA2_Score"]) / 2
    df["IA_Improvement"] = (df["IA2_Score"] - df["IA1_Score"])
    df["GPA_Trend"] = df["Final_Sem_GPA"] - df["Prev_GPA"]
    df["Academic_Risk"] = ((df["Backlog_Count"] * 0.4) + ((10 - df["Prev_GPA"]) * 0.3) + ((10 - df["IA_Average"]) * 0.3))
    df["Effort_Performance_Gap"] = df["Effort_Score"] - df["IA_Average"]
    df["Distraction_Index"] = (df["Social_Distraction"] * 0.6 + df["FreeTime"] * 0.2 + df["Romantic"] * 0.2)
    df["Discipline_Score"] = (df["AttendanceRate"] * 0.6 +(df["StudyHours"] / 4) * 0.4)
    df["Burnout_Risk"] = (df["stress_survey_score"] * 0.5 - df["sleep_hours"] * 0.3 - df["motivation_survey_score"] * 0.2)
    df["Support_Gap"] = (df["Support_Score"] - df["Effort_Score"])
    df["High_Performer"] = ((df["GPA"] > 8) & (df["AttendanceRate"] > 80)).astype(int)
    
    # --- Study Efficiency & Ability Features ---
    df["Study_Efficiency"] = df["IA_Average"] / (df["StudyHours"] + 1)
    df["Cognitive_Load"] = df["StudyHours"] + df["PartTimeJob"]*4 + df["Extracurricular"]*2
    df["Psychological_Resilience"] = df["self_efficacy_score"]*0.4 + df["motivation_survey_score"]*0.3 - df["exam_anxiety_score"]*0.3
    df["Focus_Index"] = df["Engagement_Index"] - df["Distraction_Index"]
    df["Commute_Burden"] = df["Commute_Strain"] * df["AttendanceRate"]
    
    # --- Rolling / Temporal Features ---
    df = df.sort_values(["Student_ID", "Semester_ID"])
    df["Rolling_GPA_Mean"] = df.groupby("Student_ID")["Final_Sem_GPA"].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df["Engagement_Drift"] = df.groupby("Student_ID")["Engagement_Index"].diff()
    df["Performance_Volatility"] = df.groupby("Student_ID")["Final_Sem_GPA"].transform(lambda x: x.rolling(3, min_periods=2).std())
    df["Cumulative_Academic_Risk"] = df.groupby("Student_ID")["Academic_Risk"].cumsum()
    df["Risk_Acceleration"] = df.groupby("Student_ID")["Academic_Risk"].diff()
    df["Backlog_Pressure"] = df.groupby("Student_ID")["Backlog_Count"].cumsum()
    df["Learning_Resilience"] = df["IA_Improvement"] + df["GPA_Trend"] - df["Backlog_Count"]
    
    # --- Academic Shock Features ---
    df["Academic_Shock"] = df["Rolling_GPA_Mean"] - df["Final_Sem_GPA"]
    df["Severe_Academic_Shock"] = (df["Academic_Shock"] > 1).astype(int)
    
    # Drop original test scores 
    df.drop(["TestScore_Math","TestScore_Reading","TestScore_Science"], axis=1, inplace=True, errors='ignore')
    
    return df
df_sample = feature_engineering(df_sample)

In [13]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 300000 entries, 16 to 7999974
Data columns (total 80 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Age                            300000 non-null  int64  
 1   Grade                          300000 non-null  int64  
 2   Gender                         0 non-null       float64
 3   SES_Quartile                   300000 non-null  int64  
 4   ParentalEducation              0 non-null       float64
 5   SchoolType                     0 non-null       float64
 6   GPA                            300000 non-null  float64
 7   AttendanceRate                 300000 non-null  float64
 8   StudyHours                     300000 non-null  float64
 9   InternetAccess                 300000 non-null  int64  
 10  Extracurricular                300000 non-null  int64  
 11  PartTimeJob                    300000 non-null  int64  
 12  ParentSupport                  30

In [14]:
df_sample.columns

Index(['Age', 'Grade', 'Gender', 'SES_Quartile', 'ParentalEducation',
       'SchoolType', 'GPA', 'AttendanceRate', 'StudyHours', 'InternetAccess',
       'Extracurricular', 'PartTimeJob', 'ParentSupport', 'Romantic',
       'FreeTime', 'GoOut', 'Student_ID', 'Latent_Ability',
       'study_environment_score', 'first_generation_college_flag',
       'procrastination_tendency', 'stress_survey_score', 'exam_anxiety_score',
       'motivation_survey_score', 'sleep_hours', 'self_efficacy_score',
       'goal_clarity_survey_score', 'Commute_Strain', 'Semester_ID',
       'Prev_GPA', 'Final_Sem_GPA', 'Backlog_Count', 'Assignment_Ratio',
       'Social_Distraction', 'Effort_Score', 'IA1_Score', 'IA_Improvement',
       'IA2_Score', 'Performance_Delta', 'IA_Average', 'Dropout_Probability',
       'Dropout_Flag', 'Locale_Rural', 'Locale_Suburban', 'Locale_Town',
       'Race_Black', 'Race_Hispanic', 'Race_Other', 'Race_Two-or-more',
       'Race_White', 'Archetype_elite', 'Archetype_fading_star

In [15]:
static_features = ['Age', 'Gender', 'SES_Quartile', 'ParentalEducation', 'SchoolType', 'InternetAccess', 'Race_Black', 'Race_Hispanic', 'Race_Other', 'Race_Two-or-more',
                   'Race_White', 'Locale_Rural', 'Locale_Suburban', 'Locale_Town', 'first_generation_college_flag']

historical_features = ['Prev_GPA', 'Rolling_GPA_Mean', 'GPA_Trend', 'Performance_Volatility', 'Cumulative_Academic_Risk', 'Risk_Acceleration', 'Learning_Resilience',
                       'Academic_Shock', 'Severe_Academic_Shock', 'Engagement_Drift']

current_behavioral_features = ['AttendanceRate', 'StudyHours', 'Extracurricular', 'PartTimeJob', 'ParentSupport', 'Romantic', 'FreeTime', 'GoOut', 'study_environment_score',
                      'procrastination_tendency', 'stress_survey_score', 'exam_anxiety_score', 'motivation_survey_score', 'sleep_hours', 'self_efficacy_score',
                      'goal_clarity_survey_score', 'Commute_Strain', 'Social_Distraction', 'Effort_Score', 'Support_Score', 'Distraction_Index', 'Discipline_Score',
                      'Burnout_Risk', 'Support_Gap', 'Study_Efficiency', 'Cognitive_Load', 'Psychological_Resilience', 'Focus_Index', 'Commute_Burden']

ia1_signals = ['IA1_Score', 'Assignment_Ratio', 'Backlog_Count', 'Academic_Risk', 'Effort_Performance_Gap', 'Engagement_Index', 'High_Performer']

ia2_signals = ['IA2_Score', 'IA_Average', 'IA_Improvement', 'Performance_Delta', 'Backlog_Pressure']

target_gpa = 'Final_Sem_GPA'
target_dropout = 'Dropout_Flag'

meta_columns = df_sample[['Student_ID','Semester_ID']]

In [17]:
def prepare_snapshot_datasets(df):
    X_before_ia1 = df[static_features + historical_features + current_behavioral_features]
    X_after_ia1 = df[static_features + historical_features + current_behavioral_features + ia1_signals]
    X_after_ia2 = df[static_features + historical_features + current_behavioral_features + ia1_signals + ia2_signals]
    y_gpa = df[target_gpa]
    y_dropout = df[target_dropout]
    return ( X_before_ia1, X_after_ia1, X_after_ia2, y_gpa, y_dropout)
df_before_ia, df_after_ia1, df_after_ia2, df_gpa, df_dropout = prepare_snapshot_datasets(df_sample)

In [18]:
students = df_sample['Student_ID'].unique()

train_students, test_students = train_test_split(students, test_size=0.2, random_state=42)

train_mask = df_sample['Student_ID'].isin(train_students)
test_mask = df_sample['Student_ID'].isin(test_students)

# BEFORE IA
X_before_train = df_before_ia[train_mask]
X_before_test = df_before_ia[test_mask]

# AFTER IA1
X_ia1_train = df_after_ia1[train_mask]
X_ia1_test = df_after_ia1[test_mask]

# AFTER IA2
X_ia2_train = df_after_ia2[train_mask]
X_ia2_test = df_after_ia2[test_mask]

# Targets
y_gpa_train = df_gpa[train_mask]
y_gpa_test = df_gpa[test_mask]

y_dropout_train = df_dropout[train_mask]
y_dropout_test = df_dropout[test_mask]

In [19]:
before_ia_rf_gpa = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
before_ia_rf_gpa.fit(X_before_train, y_gpa_train)

RandomForestRegressor(max_depth=10, n_jobs=-1, random_state=42)

In [20]:
pred_before = before_ia_rf_gpa.predict(X_before_test)
print("Before IA")
print("MAE:", mean_absolute_error(y_gpa_test, pred_before))
print("R2:", r2_score(y_gpa_test, pred_before))

Before IA
MAE: 0.006694125565647087
R2: 0.9998999182848691


In [37]:
corr = df_sample.corr(numeric_only=True)
corr['Final_Sem_GPA'].sort_values(ascending=False).head(30)

Final_Sem_GPA                1.000000
Rolling_GPA_Mean             0.997963
Prev_GPA                     0.980485
IA_Average                   0.892077
IA2_Score                    0.888892
IA1_Score                    0.876561
Latent_Ability               0.648412
Psychological_Resilience     0.624867
Effort_Score                 0.609167
motivation_survey_score      0.608074
goal_clarity_survey_score    0.603139
Assignment_Ratio             0.591879
self_efficacy_score          0.588956
Discipline_Score             0.588929
AttendanceRate               0.582617
StudyHours                   0.570784
Engagement_Index             0.550768
Focus_Index                  0.520904
Learning_Resilience          0.480899
IA_Improvement               0.472802
Performance_Delta            0.472802
study_environment_score      0.457946
sleep_hours                  0.456918
Archetype_elite              0.360946
Archetype_late_bloomer       0.313647
Cognitive_Load               0.241509
GPA_Trend   

In [58]:
X_before_train = X_before_train[X_before_train.columns.difference(['Rolling_GPA_Mean', 'IA_Average', 'Prev_GPA', 'IA2_Score', 'IA1_Score', 'Latent_Ability', 'Final_Sem_GPA',
                                                                   'Assignmennt_Ratio', 'AttendanceRate', 'Academic_Shock', 'Severe_Academic_Shock', 'GPA_Trend', 'Engagement_Drift',
                                                                   'Cumulative_Academic_Risk', 'Risk_Acceleration', 'Learning_Resilience', 'Performance_Volatility'])]

In [59]:
X_before_test = X_before_test[X_before_test.columns.difference(['Rolling_GPA_Mean', 'IA_Average', 'Prev_GPA', 'IA2_Score', 'IA1_Score', 'Latent_Ability', 'Final_Sem_GPA',
                                                                'Assignment_Ratio', 'AttendanceRate', 'Academic_Shock', 'Severe_Academic_Shock', 'GPA_Trend', 'Engagement_Drift',
                                                                'Cumulative_Academic_Risk', 'Risk_Acceleration', 'Learning_Resilience', 'Performance_Volatility'])]

In [60]:
X_before_train.columns

Index(['Age', 'Burnout_Risk', 'Cognitive_Load', 'Commute_Burden',
       'Commute_Strain', 'Discipline_Score', 'Distraction_Index',
       'Effort_Score', 'Extracurricular', 'Focus_Index', 'FreeTime', 'Gender',
       'GoOut', 'InternetAccess', 'Locale_Rural', 'Locale_Suburban',
       'Locale_Town', 'ParentSupport', 'ParentalEducation', 'PartTimeJob',
       'Psychological_Resilience', 'Race_Black', 'Race_Hispanic', 'Race_Other',
       'Race_Two-or-more', 'Race_White', 'Romantic', 'SES_Quartile',
       'SchoolType', 'Social_Distraction', 'StudyHours', 'Study_Efficiency',
       'Support_Gap', 'Support_Score', 'exam_anxiety_score',
       'first_generation_college_flag', 'goal_clarity_survey_score',
       'motivation_survey_score', 'procrastination_tendency',
       'self_efficacy_score', 'sleep_hours', 'stress_survey_score',
       'study_environment_score'],
      dtype='object')

In [61]:
before_ia_rf_gpa = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
before_ia_rf_gpa.fit(X_before_train, y_gpa_train)

RandomForestRegressor(max_depth=10, n_jobs=-1, random_state=42)

In [62]:
pred_before = before_ia_rf_gpa.predict(X_before_test)
print("Before IA")
print("MAE:", mean_absolute_error(y_gpa_test, pred_before))
print("R2:", r2_score(y_gpa_test, pred_before))

Before IA
MAE: 0.44415432030382906
R2: 0.8492540326723234


In [69]:
imp = pd.Series(before_ia_rf_gpa.feature_importances_, index=X_before_test.columns)
print(imp.sort_values(ascending=False).head(10))

first_generation_college_flag    0.365579
Study_Efficiency                 0.338077
StudyHours                       0.120237
Social_Distraction               0.083006
Effort_Score                     0.065561
Discipline_Score                 0.022383
Psychological_Resilience         0.000774
motivation_survey_score          0.000438
goal_clarity_survey_score        0.000401
Cognitive_Load                   0.000383
dtype: float64


In [68]:
scores = cross_val_score(before_ia_rf_gpa, X_before_test, y_gpa_test, cv=5, scoring="r2")
print(scores)
print(scores.mean())

[0.84778385 0.85021372 0.84161363 0.84706096 0.84629907]
0.8465942459369075
